# MidMamba Full Training Run (Colab)

End-to-end **Stable-Baselines3 PPO** with a **Mamba-2** feature extractor on March 2025 RTH Databento MBP-10 data.

**Architecture**:
- **Data engine**: causal/right-labeled 5-second snapshots with Pandas by default and optional PyKX/kdb+ preprocessing.
- **Market encoder**: stationary MBP-10 features only; absolute price levels are kept in simulator arrays, not policy features.
- **Simulator**: precomputed NumPy arrays plus optional Numba kernels for market-walk and passive queue-depletion fill physics.
- **Temporal model**: stacked Mamba-2 blocks with GRU fallback for CPU smoke tests.
- **Policy head**: SB3 `PPO` with `LOBMambaFeaturesExtractor`, vectorized rollouts, frame-stacked observations, and `VecNormalize`.
- **Execution control**: `action[0] = -1` waits, `0` tracks the next TWAP cumulative target, `+1` targets up to 2x TWAP progress; `action[1] < 0` posts passively before the final step, while non-negative actions cross the spread with level-walk aggressiveness. The final step becomes marketable and may walk all visible MBP-10 levels to reduce leftover inventory.

**Hardware target**: high-RAM Colab / VM + **RTX PRO 6000** or similar CUDA GPU. `NUM_ENVS` is the count of **parallel RL environments** (`SubprocVecEnv` workers), not “vCPU workers”. With large in-memory LOB chunks, **48 envs can still stress RAM** despite fork copy-on-write; if the kernel OOMs, lower `NUM_ENVS`, `CHUNK_ROWS`, or `MIN_LOADER_ROWS` first.

**Data source**: `.dbn.zst` files on Drive at `/content/drive/MyDrive/midmamba/data/march2025/`.

**Outputs** (Drive): SB3 `mamba_ppo.zip`, `mamba_ppo_vecnormalize.pkl`, `mamba_ppo.run_config.json`, metrics JSON, plots.

**Artifacts & local eval**: this notebook saves `{stem}.zip`, `{stem}_vecnormalize.pkl`, and `{stem}.run_config.json`. `scripts/evaluate_execution.py` auto-loads `{stem}_vecnormalize.pkl` beside the checkpoint, using a raw eval env and exactly one `VecNormalize` wrapper.

**Workflow**:
1. Mount Drive and copy repo to `/content/midmamba`
2. Install deps from the project (`stable-baselines3` is in `pyproject.toml`; Mamba GPU deps are built separately)
3. Load March RTH data with chunked DBN decoding, causal snapshots, and no-lookahead feature checks
4. Build vec env → create SB3 PPO → optional smoke `learn` → full `model.learn`
5. Evaluate vs Immediate / TWAP / Almgren-Chriss + SB3 policy
6. Save checkpoints and reports to Drive


In [ ]:
!pip install -U jupyter_client

In [ ]:
import multiprocessing
import os
import warnings
from contextlib import suppress

os.environ["PYTHONWARNINGS"] = "ignore::DeprecationWarning,ignore::UserWarning"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

# Colab is Linux: forkserver works well with SubprocVecEnv. Avoid force=True so re-running the cell stays safe.
with suppress(RuntimeError):
    multiprocessing.set_start_method("forkserver", force=False)

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
import os
import shutil
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/midmamba")
REPO_ROOT = Path("/content/midmamba")

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
shutil.copytree(DRIVE_ROOT, REPO_ROOT, symlinks=True)

src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.chdir(REPO_ROOT)

print(f"repo:  {REPO_ROOT}")
print(f"drive: {DRIVE_ROOT}")


In [ ]:
!pip install -U -q pip setuptools wheel ninja packaging
!pip install -q -e "/content/midmamba[dev,speed]"

# Build mamba-ssm + causal-conv1d. For Blackwell/RTX PRO 6000, compute capability is commonly 12.0.
# If wheels/builds fail, run `print(torch.cuda.get_device_capability(0))` and adjust TORCH_CUDA_ARCH_LIST.
import os

os.environ["TORCH_CUDA_ARCH_LIST"] = "12.0"
!pip install -q "causal-conv1d>=1.4.0" --no-build-isolation
!pip install -q "mamba-ssm>=2.2.0" --no-build-isolation

import stable_baselines3
import torch

print(f"torch {torch.__version__}  CUDA {torch.version.cuda}")
print(f"stable-baselines3 {stable_baselines3.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"compute capability: {torch.cuda.get_device_capability(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)
    print(f"VRAM: {vram / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected -- set BACKEND='gru' for a CPU smoke run")


In [ ]:
# -- All tunables in one place -------------------------------------------------

import os
from contextlib import suppress
from pathlib import Path

import torch

BACKEND        = "mamba"       # "mamba" for CUDA, "gru" for CPU fallback
D_MODEL        = 192
N_LAYERS       = 3
SEQ_LEN        = 128
SPATIAL_STEM   = True          # bid/ask-aware LOBSpatialStem vs flat linear
DROPOUT        = 0.1
NET_ARCH       = dict(pi=[256, 256], vf=[256, 256])  # SB3 actor/critic MLP after the Mamba feature extractor
USE_NUMBA      = True           # optional compiled simulator kernels for fill physics
USE_BF16       = True           # BF16 autocast around SB3 policy forward/evaluation calls
USE_TORCH_COMPILE = False       # optional: compile only the LOB backbone, not the full SB3 policy
TORCH_COMPILE_MODE = "reduce-overhead"
NUM_ENVS       = 48            # SB3 parallel env processes; reduce if RAM spikes
ROLLOUT_STEPS  = 512           # 512 x 48 envs = 24K steps per update
UPDATES        = 2000
N_EPOCHS       = 4             # SB3 PPO optimizer passes over each rollout buffer
BATCH_SIZE     = 4096          # SB3 minibatch size; auto-adjusted to divide rollout buffer
CLIP_COEF      = 0.1
MAX_GRAD_NORM  = 0.5
ENTROPY_COEF   = 0.0003
TARGET_KL      = 0.02          # set None to disable SB3 KL early stop
EXECUTION_STEPS   = 360        # 360 x 5s = 30-minute horizon
PARENT_QUANTITY   = 100_000.0
LR             = 5e-5
LR_SCHEDULE    = "cosine"      # "constant", "linear", or "cosine"
LR_WARMUP      = 10            # warmup length in rollout units; total warmup = LR_WARMUP * ROLLOUT_STEPS * NUM_ENVS
GAMMA          = 0.995
GAE_LAMBDA     = 0.95
FILL_MODEL     = "random"      # per-episode fill-model randomization
SIDE           = "buy"
SEED           = 1

# Execution-control reminder:
# action[0] = -1 waits, 0 tracks TWAP cumulative target, +1 targets up to 2x TWAP progress.
# action[1] < 0 posts passively before the final step; >= 0 crosses the spread.

# -- Multi-objective volatility-scaled reward --------------------------------
BETA_IS         = 1.0          # implementation shortfall is the main objective
BETA_SCHEDULE   = 0.1          # soft guide around TWAP, not the main objective
BETA_COMPLETION = 1.0          # volatility-scaled leftover risk
TERMINAL_PENALTY_BPS = 100.0   # hard terminal leftover-inventory penalty in bps
REWARD_CLIP     = 5.0          # 0 disables clipping
NORM_REWARD     = True         # normalize discounted returns for long execution episodes

# -- Transaction costs --------------------------------------------------------
TAKER_FEE_BPS    = 2.0
MAKER_REBATE_BPS = 0.5

# -- Evaluation ---------------------------------------------------------------
CHECKPOINT_EVERY = 100
EVAL_EPISODES    = 200
TWAP_SLICES      = 100
AC_RISK_AVERSION     = 1e-6
AC_VOLATILITY        = 0.02
AC_TEMPORARY_IMPACT  = 1.0

# -- Data loading: training ---------------------------------------------------
DBN_GLOB       = "data/march2025/*.dbn.zst"
CHUNK_ROWS     = 500_000
MIN_LOADER_ROWS = 1_000
MAX_CHUNKS     = None
RTH_ONLY       = True
RTH_START      = "09:30:00"
RTH_END        = "16:00:00"
RESAMPLE_FREQ  = "5s"         # 5-second fixed cadence
SNAPSHOT_BACKEND = "pandas"    # "pykx" enables optional kdb+ snapshot preprocessing

# -- Data loading: evaluation -------------------------------------------------
EVAL_DBN_GLOB  = "data/october2025/*.dbn.zst"

# -- Output paths (Drive-backed) ---------------------------------------------
DRIVE_OUT       = Path("/content/drive/MyDrive/midmamba")
SB3_MODEL_PATH = DRIVE_OUT / "checkpoints" / "mamba_ppo.zip"
SB3_VECNORM_PATH = DRIVE_OUT / "checkpoints" / "mamba_ppo_vecnormalize.pkl"
RESULTS_DIR     = DRIVE_OUT / "results"

SB3_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TOTAL_TIMESTEPS = UPDATES * ROLLOUT_STEPS * NUM_ENVS

# Many parallel envs: avoid BLAS/OpenMP oversubscription.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ["MIDMAMBA_USE_NUMBA"] = "1" if USE_NUMBA else "0"
with suppress(Exception):
    torch.set_num_threads(1)
with suppress(Exception):
    torch.set_float32_matmul_precision("high")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if BACKEND == "mamba" and DEVICE.type != "cuda":
    raise RuntimeError("BACKEND='mamba' requires CUDA. Set BACKEND='gru' for CPU smoke tests.")
if SNAPSHOT_BACKEND not in {"pandas", "pykx"}:
    raise ValueError("SNAPSHOT_BACKEND must be 'pandas' or 'pykx'")
if USE_NUMBA:
    import numba
    print(f"numba {numba.__version__}")
if SNAPSHOT_BACKEND == "pykx":
    try:
        import pykx
    except ImportError as exc:
        raise RuntimeError("SNAPSHOT_BACKEND='pykx' requires `pip install pykx` and a configured kdb+ license") from exc
    if not os.environ.get("QLIC"):
        print("WARNING: SNAPSHOT_BACKEND='pykx' usually requires QLIC to point at your kdb+ license directory")
print(f"device={DEVICE}  backend={BACKEND}  total_timesteps={TOTAL_TIMESTEPS:,}")
print(f"snapshot_backend={SNAPSHOT_BACKEND}  causal_right_labeled=True")
print(f"numba_kernels={USE_NUMBA}  bf16_autocast={USE_BF16}  torch_compile_backbone={USE_TORCH_COMPILE}")


In [ ]:
dbn_files = sorted(REPO_ROOT.glob(DBN_GLOB))
print(f"found {len(dbn_files)} DBN files")
if dbn_files:
    print(f"  first: {dbn_files[0].name}")
    print(f"  last:  {dbn_files[-1].name}")
else:
    raise FileNotFoundError(
        f"no .dbn.zst files matched {DBN_GLOB} under {REPO_ROOT}.\n"
        "Make sure your Drive contains the data at /MyDrive/midmamba/data/march2025/"
    )

In [ ]:
!cd /content/midmamba && python -m pytest tests -q

## Training

Run cells in order after a kernel restart: **Drive → install → tunables → DBN list → tests → loader/env → PPO model → smoke (optional) → full learn → plots → evaluation**.

The smoke cell continues from the same `model` object. For a fresh full run, re-run the **loader/env** and **PPO model** cells, then skip smoke.

The test cell includes causal snapshot, no-lookahead, and stationary-feature checks; keep it in the run path whenever preprocessing changes.


In [ ]:
from midmamba.data import MBP10WindowLoader
from midmamba.env import MidMambaExecutionEnv
from midmamba.rl import build_vec_env


def _progress(info):
    print(
        f"  file={info.get('file_index', 1)} "
        f"chunk={info['chunk_index']} "
        f"decoded={info['decoded_rows']:,} "
        f"kept={info['kept_rows']:,}",
        end="\r",
    )


print("loading DBN data (chunked)...")
loader = MBP10WindowLoader.from_dbn_files_chunks(
    dbn_files,
    chunk_rows=CHUNK_ROWS,
    min_rows=MIN_LOADER_ROWS,
    max_chunks=MAX_CHUNKS,
    resample_freq=RESAMPLE_FREQ,
    snapshot_backend=SNAPSHOT_BACKEND,
    rth_start=RTH_START if RTH_ONLY else None,
    rth_end=RTH_END if RTH_ONLY else None,
    seed=SEED,
    progress_callback=_progress,
)
print(f"\nloader ready: {loader.n_rows:,} rows, {loader.n_features} features")
print(f"session gaps detected: {len(loader._session_ends)}")
print(f"feature sample: {loader.feature_names[:5]} ...")
raw_price_features = [
    c for c in loader.feature_names
    if c in {"mid", "micro_price", "weighted_mid"} or ("_px_" in c and not c.endswith("_rel_mid"))
]
if raw_price_features:
    raise RuntimeError(f"absolute price features leaked into policy input: {raw_price_features}")
required_relative_features = {"micro_price_rel_mid", "weighted_mid_rel_mid"}
missing_relative = sorted(required_relative_features - set(loader.feature_names))
if missing_relative:
    raise RuntimeError(f"missing stationary relative price features: {missing_relative}")
print(f"snapshot backend={SNAPSHOT_BACKEND}; snapshots are causal/right-labeled; policy features use no future rows")
print(f"simulator backend={'numba' if USE_NUMBA else 'numpy'}; passive queue-depletion flows are precomputed")

_reward_kwargs = dict(
    beta_is=BETA_IS,
    beta_schedule=BETA_SCHEDULE,
    beta_completion=BETA_COMPLETION,
    reward_clip=REWARD_CLIP,
    taker_fee_bps=TAKER_FEE_BPS,
    maker_rebate_bps=MAKER_REBATE_BPS,
    terminal_penalty_bps=TERMINAL_PENALTY_BPS,
)

vec_env = build_vec_env(
    loader,
    n_envs=NUM_ENVS,
    stack_size=SEQ_LEN,
    seed=SEED,
    execution_steps=EXECUTION_STEPS,
    parent_quantity=PARENT_QUANTITY,
    side=SIDE,
    fill_model=FILL_MODEL,
    gamma=GAMMA,
    norm_obs=True,
    norm_reward=NORM_REWARD,
    reward_kwargs=_reward_kwargs,
    use_subproc=NUM_ENVS > 1,
)
print(f"vec_env: {NUM_ENVS} envs + FrameStackObservation(seq={SEQ_LEN}) + exactly one VecNormalize")
print(f"  obs_space={vec_env.observation_space.shape}  action_space={vec_env.action_space.shape}")
print(f"  reward: beta_is={BETA_IS} beta_schedule={BETA_SCHEDULE} beta_completion={BETA_COMPLETION} terminal={TERMINAL_PENALTY_BPS} clip={REWARD_CLIP}")

single_env = MidMambaExecutionEnv(
    loader,
    execution_steps=EXECUTION_STEPS,
    initial_inventory=PARENT_QUANTITY,
    side=SIDE,
    fill_model=FILL_MODEL,
    **_reward_kwargs,
)
n_obs = int(single_env.observation_space.shape[0])


In [ ]:
from midmamba.ppo_rollout import best_batch_size_for_rollout
from midmamba.rl import (
    AutocastActorCriticPolicy,
    compile_sb3_backbone,
    execution_obs_feature_names,
    make_lr_schedule,
    make_ppo,
    midmamba_policy_kwargs,
    stacked_observation_space,
)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

feature_names_ext = execution_obs_feature_names(loader.feature_names) if SPATIAL_STEM else None
obs_space = stacked_observation_space(n_obs, SEQ_LEN)
policy_kwargs = midmamba_policy_kwargs(
    observation_space=obs_space,
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
    backend=BACKEND,
    spatial_stem=SPATIAL_STEM,
    feature_names=feature_names_ext,
    net_arch=NET_ARCH,
    autocast_enabled=USE_BF16,
    autocast_device_type=DEVICE.type,
    autocast_dtype="bfloat16",
)

_rollout_buf = int(ROLLOUT_STEPS) * int(NUM_ENVS)
_batch = best_batch_size_for_rollout(_rollout_buf, min(int(BATCH_SIZE), _rollout_buf))

lr_sched = make_lr_schedule(
    LR,
    total_timesteps=TOTAL_TIMESTEPS,
    warmup_timesteps=int(LR_WARMUP * ROLLOUT_STEPS * NUM_ENVS),
    schedule=LR_SCHEDULE,
)

policy_class = AutocastActorCriticPolicy if USE_BF16 else "MlpPolicy"

model = make_ppo(
    vec_env,
    policy=policy_class,
    learning_rate=lr_sched,
    n_steps=ROLLOUT_STEPS,
    batch_size=_batch,
    n_epochs=N_EPOCHS,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    clip_range=CLIP_COEF,
    ent_coef=ENTROPY_COEF,
    max_grad_norm=MAX_GRAD_NORM,
    target_kl=TARGET_KL,
    seed=SEED,
    device=str(DEVICE),
    policy_kwargs=policy_kwargs,
    verbose=1,
)
compiled_backbone = compile_sb3_backbone(
    model,
    enabled=bool(USE_TORCH_COMPILE and DEVICE.type == "cuda"),
    mode=TORCH_COMPILE_MODE,
)
print(f"SB3 PPO batch_size={_batch} (rollout buffer={_rollout_buf})")
print(f"BF16 autocast={USE_BF16}  compiled_backbone={compiled_backbone}")
print(f"policy parameter dtype={next(model.policy.parameters()).dtype}")
print(f"SB3 policy: {model.policy}")

run_config = {
    "backend": BACKEND,
    "d_model": D_MODEL,
    "n_layers": N_LAYERS,
    "spatial_stem": SPATIAL_STEM,
    "use_numba": USE_NUMBA,
    "use_bf16": USE_BF16,
    "use_torch_compile": USE_TORCH_COMPILE,
    "torch_compile_mode": TORCH_COMPILE_MODE,
    "dropout": DROPOUT,
    "net_arch": NET_ARCH,
    "n_features": n_obs,
    "seq_len": SEQ_LEN,
    "norm_obs": True,
    "num_envs": NUM_ENVS,
    "rollout_steps": ROLLOUT_STEPS,
    "updates": UPDATES,
    "n_epochs": N_EPOCHS,
    "batch_size": BATCH_SIZE,
    "batch_size_effective": _batch,
    "max_grad_norm": MAX_GRAD_NORM,
    "execution_steps": EXECUTION_STEPS,
    "parent_quantity": PARENT_QUANTITY,
    "lr": LR,
    "lr_schedule": LR_SCHEDULE,
    "lr_warmup": LR_WARMUP,
    "clip_coef": CLIP_COEF,
    "target_kl": TARGET_KL,
    "entropy_coef": ENTROPY_COEF,
    "gamma": GAMMA,
    "gae_lambda": GAE_LAMBDA,
    "fill_model": FILL_MODEL,
    "side": SIDE,
    "seed": SEED,
    "loader_rows": loader.n_rows,
    "snapshot_backend": SNAPSHOT_BACKEND,
    "resample_freq": RESAMPLE_FREQ,
    "snapshot_clock": "right_labeled_causal",
    "feature_contract": "stationary_relative_no_future",
    "simulator_backend": "numba" if USE_NUMBA else "numpy",
    "non_stationary_feature_guard": True,
    "device": str(DEVICE),
    "beta_is": BETA_IS,
    "beta_schedule": BETA_SCHEDULE,
    "beta_completion": BETA_COMPLETION,
    "terminal_penalty_bps": TERMINAL_PENALTY_BPS,
    "reward_clip": REWARD_CLIP,
    "taker_fee_bps": TAKER_FEE_BPS,
    "maker_rebate_bps": MAKER_REBATE_BPS,
    "norm_reward": NORM_REWARD,
    "total_timesteps": TOTAL_TIMESTEPS,
    "action_semantics": "size action is TWAP-relative cumulative target: -1 wait, 0 TWAP, +1 up to 2x TWAP",
    "sb3": "stable-baselines3",
}


In [ ]:
import time

from midmamba.rl import SB3RolloutLoggerCallback

SMOKE_TIMESTEPS = 50 * ROLLOUT_STEPS * NUM_ENVS
smoke_cb = SB3RolloutLoggerCallback()
t0 = time.time()
print(f"smoke run: {SMOKE_TIMESTEPS} timesteps, rollout={ROLLOUT_STEPS}, num_envs={NUM_ENVS}")
model.learn(total_timesteps=SMOKE_TIMESTEPS, callback=smoke_cb, progress_bar=True)
smoke_history = smoke_cb.rows
print(f"\nsmoke logged {len(smoke_history)} rollout updates  elapsed={time.time()-t0:.0f}s")
print("full run continues from current SB3 weights (same model object)")


In [ ]:
import json

from stable_baselines3.common.callbacks import CallbackList, CheckpointCallback

from midmamba.rl import SB3RolloutLoggerCallback, save_sb3_checkpoint

history_cb = SB3RolloutLoggerCallback()
callbacks = [history_cb]
if USE_TORCH_COMPILE:
    print("intermediate CheckpointCallback disabled with torch.compile; final save unwraps the backbone")
else:
    ck_every = max(int(CHECKPOINT_EVERY * ROLLOUT_STEPS * NUM_ENVS), ROLLOUT_STEPS * NUM_ENVS)
    ck_cb = CheckpointCallback(
        save_freq=ck_every,
        save_path=str(SB3_MODEL_PATH.parent),
        name_prefix=SB3_MODEL_PATH.stem + "_sb3_",
    )
    callbacks.append(ck_cb)
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=CallbackList(callbacks),
    progress_bar=True,
)
history = history_cb.rows

save_sb3_checkpoint(model, vec_env, model_path=str(SB3_MODEL_PATH), vecnorm_path=str(SB3_VECNORM_PATH))
SB3_MODEL_PATH.with_suffix(".run_config.json").write_text(json.dumps(run_config, indent=2, default=str))

metrics_path = RESULTS_DIR / "ppo_training_metrics.json"
metrics_path.write_text(json.dumps({"config": run_config, "history": history}, indent=2, default=str))
print(f"checkpoint: {SB3_MODEL_PATH}")
print(f"vecnorm:    {SB3_VECNORM_PATH}")
print(f"metrics:    {metrics_path}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not history:
    print("no history rows — skip plot")
else:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    upd = [r["update"] for r in history]
    rew_key = "rollout/ep_rew_mean"
    if rew_key not in history[0]:
        rew_key = next((k for k in history[0] if "rew" in k.lower()), None)
    if rew_key:
        axes[0, 0].plot(upd, [r.get(rew_key, np.nan) for r in history])
        axes[0, 0].set_title("Episode reward mean (SB3)")
    axes[0, 0].set_xlabel("Rollout")
    axes[0, 0].grid(True)

    kl_key = next((k for k in history[0] if "kl" in k.lower()), None)
    if kl_key:
        axes[0, 1].plot(upd, [r.get(kl_key, np.nan) for r in history])
        axes[0, 1].set_title(kl_key)
        axes[0, 1].grid(True)

    pol_key = next((k for k in history[0] if "policy" in k.lower() and "loss" in k.lower()), None)
    if pol_key:
        axes[1, 0].plot(upd, [r.get(pol_key, np.nan) for r in history])
        axes[1, 0].set_title(pol_key)
        axes[1, 0].grid(True)

    vf_key = next((k for k in history[0] if "value" in k.lower() and "loss" in k.lower()), None)
    if vf_key:
        axes[1, 1].plot(upd, [r.get(vf_key, np.nan) for r in history])
        axes[1, 1].set_title(vf_key)
        axes[1, 1].grid(True)

    plt.tight_layout()
    plot_path = RESULTS_DIR / "training_curves.png"
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f"saved: {plot_path}")


## Evaluation

In [ ]:
# Load out-of-sample evaluation data (separate from training)
eval_dbn_files = sorted(REPO_ROOT.glob(EVAL_DBN_GLOB))
if eval_dbn_files:
    print(f"loading eval data: {len(eval_dbn_files)} files from {EVAL_DBN_GLOB}")
    eval_loader = MBP10WindowLoader.from_dbn_files_chunks(
        eval_dbn_files,
        chunk_rows=CHUNK_ROWS,
        min_rows=max(EXECUTION_STEPS, TWAP_SLICES + 1),
        resample_freq=RESAMPLE_FREQ,
        snapshot_backend=SNAPSHOT_BACKEND,
        rth_start=RTH_START if RTH_ONLY else None,
        rth_end=RTH_END if RTH_ONLY else None,
        seed=SEED + 1000,
    )
    print(f"eval loader: {eval_loader.n_rows:,} rows (out-of-sample)")
else:
    print(f"WARNING: no eval files found at {EVAL_DBN_GLOB} -- falling back to training data (in-sample)")
    eval_loader = loader

In [ ]:
from midmamba.eval import run_almgren_chriss_execution, run_immediate_execution, run_twap_execution, twap_effective_slices

_, eval_raw_lob = eval_loader.sample_window(max(EXECUTION_STEPS, TWAP_SLICES + 1))

immediate = run_immediate_execution(eval_raw_lob, side=SIDE, parent_quantity=PARENT_QUANTITY)
twap = run_twap_execution(eval_raw_lob, side=SIDE, parent_quantity=PARENT_QUANTITY, n_slices=TWAP_SLICES)
almgren = run_almgren_chriss_execution(
    eval_raw_lob,
    side=SIDE,
    parent_quantity=PARENT_QUANTITY,
    n_slices=TWAP_SLICES,
    risk_aversion=AC_RISK_AVERSION,
    volatility=AC_VOLATILITY,
    temporary_impact=AC_TEMPORARY_IMPACT,
)

print(f"{'Baseline':<20} {'IS (bps)':>10} {'Filled':>10} {'Remaining':>12} {'Reward':>10}")
print("-" * 65)
for b in [immediate, twap, almgren]:
    print(
        f"{b.name:<20} {b.implementation_shortfall_bps:>10.2f} "
        f"{b.filled_qty:>10.0f} {b.remaining_inventory:>12.0f} {b.total_reward:>10.2f}"
    )


In [ ]:
import numpy as np
from stable_baselines3 import PPO

from midmamba.eval import default_vecnormalize_path, normalize_vec_step_output, run_policy_evaluation
from midmamba.rl import load_eval_vec_env

# Eval uses proportional fill for a stable IS estimate; training used FILL_MODEL (often "random").
# load_eval_vec_env builds a raw eval vec env first, then applies exactly one VecNormalize wrapper.
_vecnorm_pkl = default_vecnormalize_path(SB3_MODEL_PATH)
if _vecnorm_pkl is None and SB3_VECNORM_PATH.is_file():
    _vecnorm_pkl = SB3_VECNORM_PATH

eval_vec = load_eval_vec_env(
    loader=eval_loader,
    stack_size=SEQ_LEN,
    seed=SEED + 123,
    execution_steps=EXECUTION_STEPS,
    parent_quantity=PARENT_QUANTITY,
    side=SIDE,
    fill_model="proportional",
    gamma=GAMMA,
    norm_obs=True,
    norm_reward=NORM_REWARD,
    reward_kwargs=_reward_kwargs,
    vecnorm_path=str(_vecnorm_pkl) if _vecnorm_pkl is not None else None,
    training_vec=vec_env,
    trust_vecnormalize=True,  # this notebook loads stats it just trained/saved
)
# PPO.load uses pickle-style deserialization; only load checkpoints you created or otherwise trust.
eval_model = PPO.load(str(SB3_MODEL_PATH), env=eval_vec, device=str(DEVICE), print_system_info=False)

eval_result = run_policy_evaluation(eval_model, eval_vec, n_episodes=EVAL_EPISODES, deterministic=True)
eval_result.print_summary()
policy_shortfalls = eval_result.shortfalls_bps

print(f"\n{'Strategy':<20} {'IS (bps)':>10}")
print("-" * 32)
print(f"{'Immediate':<20} {immediate.implementation_shortfall_bps:>10.2f}")
print(f"{'TWAP':<20} {twap.implementation_shortfall_bps:>10.2f}")
print(f"{'Almgren-Chriss':<20} {almgren.implementation_shortfall_bps:>10.2f}")
print(f"{'Policy (mean)':<20} {np.mean(policy_shortfalls):>10.2f}")


In [ ]:
import matplotlib.pyplot as plt

from midmamba.env import MBP10ExecutionEnv

reset_out = eval_vec.reset()
obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out
policy_traj = []
while True:
    action, _ = eval_model.predict(obs, deterministic=True)
    step_out = eval_vec.step(action)
    obs, _r, dones, infos = normalize_vec_step_output(step_out)
    info = infos[0] if isinstance(infos, list | tuple) else infos
    policy_traj.append(info)
    if bool(dones[0]):
        break

_twap_plot_slices = twap_effective_slices(eval_raw_lob, TWAP_SLICES)
twap_env = MBP10ExecutionEnv(
    eval_raw_lob,
    side=SIDE,
    parent_quantity=PARENT_QUANTITY,
    child_fraction=1.0 / float(_twap_plot_slices),
    end_index=_twap_plot_slices,
)
_, twap_info = twap_env.reset()
twap_traj = [twap_info]
while True:
    _, _, terminated, truncated, twap_info = twap_env.step(1)
    twap_traj.append(twap_info)
    if terminated or truncated:
        break

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

for name, traj in [("Policy", policy_traj), ("TWAP", twap_traj)]:
    steps = [t.get("step", t.get("row", i)) for i, t in enumerate(traj)]
    inv = [t.get("inventory", t.get("remaining_inventory")) for t in traj]
    is_bps = [t["implementation_shortfall_bps"] for t in traj]
    axes[0].plot(steps, inv, label=name)
    axes[1].plot(steps, is_bps, label=name)

for name, traj in [("Policy", policy_traj), ("TWAP", twap_traj)]:
    p_steps = [t.get("step", t.get("row", i)) for i, t in enumerate(traj)]
    mids = [t["mid_now"] for t in traj]
    axes[2].plot(p_steps, mids, linestyle="--" if name == "TWAP" else "-", alpha=0.85, label=f"mid ({name})")

axes[0].set_ylabel("Inventory")
axes[0].set_title("Execution Inventory Trajectory")
axes[0].legend()
axes[0].grid(True)

axes[1].set_ylabel("IS (bps)")
axes[1].set_title("Cumulative Implementation Shortfall")
axes[1].legend()
axes[1].grid(True)

axes[2].set_ylabel("Price")
axes[2].set_title("Market Mid Price")
axes[2].set_xlabel("Step")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
traj_path = RESULTS_DIR / "execution_trajectory.png"
plt.savefig(traj_path, dpi=150)
plt.show()
print(f"saved: {traj_path}")


In [ ]:
import json

report = {
    "config": run_config,
    "baselines": {
        "immediate": immediate.to_dict(),
        "twap": twap.to_dict(),
        "almgren_chriss": almgren.to_dict(),
    },
    "policy": eval_result.summary(),
    "artifact_paths": {
        "checkpoint": str(SB3_MODEL_PATH),
        "vecnormalize": str(SB3_VECNORM_PATH),
        "training_metrics": str(RESULTS_DIR / "ppo_training_metrics.json"),
        "train_plot": str(RESULTS_DIR / "training_curves.png"),
        "trajectory_plot": str(RESULTS_DIR / "execution_trajectory.png"),
    },
}

report_path = RESULTS_DIR / "colab_full_eval.json"
report_path.write_text(json.dumps(report, indent=2, default=str))

print("Drive outputs:")
print(f"  checkpoint:  {SB3_MODEL_PATH}")
print(f"  vecnorm:     {SB3_VECNORM_PATH}")
print(f"  training:    {RESULTS_DIR / 'ppo_training_metrics.json'}")
print(f"  eval report: {report_path}")
print(f"  train plot:  {RESULTS_DIR / 'training_curves.png'}")
print(f"  traj plot:   {RESULTS_DIR / 'execution_trajectory.png'}")
